In [1]:
import pandas as pd
import yfinance as yf

In [2]:
catalyst_features = pd.read_csv("datasets/catalyst_ds.csv")

In [3]:
final_tickers = catalyst_features['company_ticker'].dropna().unique()

In [4]:
# pull company financial features for each ticker

def getFinancialInfo(tickers):
    financial_data = []
    for ticker in tickers:
        company = yf.Ticker(ticker)
        row = {"ticker": ticker}
        for statement in [company.financials, company.balance_sheet, company.cashflow]:
            for col in statement.columns:
                if col.year == 2024:
                    row.update(statement[col])
        financial_data.append(row)
    financial_data = pd.DataFrame(financial_data).reindex(columns = 
            ['ticker', 
            'Total Revenue',  
            'Operating Revenue',
            'Cash And Cash Equivalents',
            'Total Debt',
            'Capital Expenditure'])
    return financial_data

In [5]:
df = getFinancialInfo(final_tickers)

/Applications/anaconda4/anaconda3/lib/python3.12/site-packages/yfinance/scrapers/fundamentals.py:131: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/Applications/anaconda4/anaconda3/lib/python3.12/site-packages/yfinance/scrapers/fundamentals.py:131: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/Applications/anaconda4/anaconda3/lib/python3.12/site-packages/yfinance/scrapers/fundamentals.py:131: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/Applications/anaconda4/anaconda3/lib/python3.12/site-packages/yfinance/scrapers/fundamentals.py:131: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now(

In [6]:
# company drug portfolio size
catalyst_features_copy=catalyst_features.copy()
catalyst_features_copy['date'] = pd.to_datetime(catalyst_features_copy['date'])
drug_count = catalyst_features_copy[catalyst_features_copy['date'].dt.year <= 2024].groupby('company_ticker')['drug_name'].nunique()

# map back to the df company table
df['drug_count'] = df['ticker'].map(drug_count).fillna(0).astype(int)

In [7]:
df.to_csv("datasets/company_ds.csv", index = False)